# This notebook calculate the ratio of shear stress to effective normal stress, which will be used as continuous responses in the subsequent DGSA sensitivity analysis

# 3. Calculate the ratio of shear stress to effective normal stress

## perform fault slip analysis on each case

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from fault_slip_analysis import tau_sigma_ratio
from tqdm import tqdm

# set up path
base_path = Path('.')
# user inputs
name_prefix = '250922'; n_cases = 90; n_faults = 12
ratio_by_fault = np.full((107,117,5,6), np.nan)
fault_info = pd.read_csv(base_path/'data'/'raw'/'fault_strike_dip.csv')
coor_fault = np.load(base_path/'data'/'coor_fault'/'JD_Sula_2025_gmc_coor&fault_reservoir.npy')
parameters = pd.read_csv(base_path/'data'/'params_responses'/f'{name_prefix}_CMG_parameters.csv')

save_folder_path = base_path/'data'/f'{name_prefix}_ratio'
save_folder_path.mkdir(parents=True, exist_ok=True)

# run analysis
# for case_num in tqdm(range(50,51), desc='Running stress-based fault slip analysis'):
for case_num in tqdm(range(1,n_cases+1), desc='Calculating the ratio of shear stress to effective normal stress'):
    # load principal stress arrays
    SH = np.load(base_path/'data'/f"{name_prefix}_gmc"/f'case{case_num}_STRESMXP.npy')
    Sh = np.load(base_path/'data'/f"{name_prefix}_gmc"/f'case{case_num}_STRESMNP.npy')
    Sv = np.load(base_path/'data'/f"{name_prefix}_gmc"/f'case{case_num}_STRESINT.npy')
    # extract the azimuth of the maximum horizontal stress from the parameter dataframe
    SH_azi = parameters.loc[parameters["case_name"] == f'case{case_num}','SH_azi_deg'].iloc[0]

    for fault_id in range(n_faults):
        row = fault_info.loc[fault_info['fault_id'] == fault_id].iloc[0]
        fault_slip = tau_sigma_ratio(
            SH = SH,
            Sh = Sh,
            Sv = Sv,
            SH_azi = SH_azi,
            fault_strike = row['fault_strike_deg'],
            fault_dip = row['dip_angle_deg']
            )

        # save analysis to the specific fault 
        fault_id_mask = (coor_fault[:,:,:,3] == fault_id)
        ratio_by_fault[fault_id_mask] = fault_slip[fault_id_mask]

    # save results for one case
    np.save(save_folder_path/f'case{case_num}_ratio.npy', ratio_by_fault)

print("\nFinished analyzing all cases.")

Calculating the ratio of shear stress to effective normal stress: 100%|██████████| 90/90 [02:17<00:00,  1.52s/it]


Finished analyzing all cases.


In [6]:
arr = np.load('data/250922_ratio/case10_ratio.npy')
arr.shape
print(np.nanmean(arr))
print(np.nanmax(arr))

0.21602929530127218
0.5151796395298871


## combine all cases in a numpy array (n_cases,n_faults,n_times) containing total number of slipped cells

In [21]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

# user inputs
name_prefix = '250922'; n_cases = 90; n_faults = 12; n_times = 6
fault_info = pd.read_csv('data/raw/fault_strike_dip.csv')
coor_fault = np.load('data/coor_fault/JD_Sula_2025_gmc_coor&fault_reservoir.npy')
ratio_combined = np.full((n_cases,n_faults,n_times), np.nan)

# set up paths
base_path = Path('.')
ratio_folder = base_path/'data'/f'{name_prefix}_ratio'

for case_num in tqdm(range(1,n_cases+1), desc='Combining FSA results for all cases'):
    FSA = np.load(ratio_folder/f'case{case_num}_ratio.npy')

    for fault_id in range(0,n_faults):
        # save analysis to the specific fault 
        fault_id_mask = (coor_fault[:,:,:,3] == fault_id)
        ratio_combined[case_num-1,fault_id,:] = np.nanmean(FSA[fault_id_mask],axis=0)

# save
np.save(base_path/'data'/f'{name_prefix}_ratio_combined.npy',ratio_combined)

print("\nFinished combining FSA results for all cases to one numpy array.")
# np.savetxt(base_path/'data'/f'{save_file_prefix}_FSA_combined.csv',FSA_combined,delimiter=",",fmt="%.4f")

Combining FSA results for all cases: 100%|██████████| 90/90 [00:00<00:00, 373.96it/s]


Finished combining FSA results for all cases to one numpy array.


In [19]:
print(ratio_combined.shape)
print(ratio_combined[4,:,:])

(90, 12, 6)
[[0.34895761 0.36434273 0.37828266 0.37899974 0.37767729 0.37716016]
 [0.05522462 0.05922558 0.06462135 0.06530161 0.06485398 0.06466417]
 [0.00401327 0.0041259  0.00424222 0.00423377 0.00422474 0.00422092]
 [0.01854803 0.01935027 0.02007501 0.01994695 0.0199016  0.01987793]
 [0.48610005 0.53507422 0.57916214 0.56772719 0.56558668 0.56412963]
 [0.11156946 0.11529267 0.11972291 0.12019932 0.11984256 0.1196941 ]
 [0.52377695 0.55534075 0.5999007  0.60764646 0.60443518 0.60287019]
 [0.03099954 0.03195158 0.03286147 0.03285704 0.03280083 0.03276921]
 [0.01954197 0.02061532 0.02211751 0.02237797 0.02226288 0.02221074]
 [0.05510045 0.05665393 0.05787397 0.05748448 0.05742385 0.05738382]
 [0.09902118 0.10254373 0.10644178 0.10651418 0.1061946  0.10606565]
 [0.29558477 0.30582726 0.31643851 0.31601823 0.31524764 0.31489797]]


# calculate responses for DGSA sensitivity analysis

per fault

In [2]:
import numpy as np

name_prefix = '250922'; fault_id = [i for i in range(12)]; year = 2050; year_list = [2030, 2040, 2050, 2060, 2550, 3050]
mean_stress_ratio = np.load(f'data/{name_prefix}_mean_stress_ratio.npy')
print(mean_stress_ratio.shape)
resp = mean_stress_ratio[:,fault_id,year_list.index(year)]

# create headers
# Define column names in a list
column_headers = [f'fault{id}' for id in fault_id]
# Join the list of names into a single string using your delimiter
header_string = ','.join(column_headers)
print(header_string)
# np.savetxt(f'data/params_responses/{name_prefix}_CMG_responses_fault{fault_id}.csv',resp,delimiter=',',fmt='%d',header='FSA',comments='')
np.savetxt(f'data/params_responses/{name_prefix}_CMG_responses_ratio_max.csv',resp,delimiter=',',fmt='%.3f',header=header_string,comments='')
resp

(90, 12, 6)
fault0,fault1,fault2,fault3,fault4,fault5,fault6,fault7,fault8,fault9,fault10,fault11


array([[0.42402952, 0.12478785, 0.05940409, ..., 0.00694631, 0.03796946,
        0.24314103],
       [0.46425411, 0.1675175 , 0.10319596, ..., 0.0503298 , 0.00507834,
        0.19947788],
       [0.49634135, 0.24738478, 0.1823186 , ..., 0.13250822, 0.08965535,
        0.10401845],
       ...,
       [0.52514008, 0.20192445, 0.13087505, ..., 0.07471225, 0.02673833,
        0.19098038],
       [0.46398468, 0.14321447, 0.07075281, ..., 0.01544237, 0.03170564,
        0.2489682 ],
       [0.51282367, 0.23775139, 0.17230582, ..., 0.12028528, 0.07567245,
        0.12576087]])